# Rubric review on a DYNAMO VM — FPPG Project Deliverables Checklist

Runs on a `dynamo02`–`dynamo08` Windows VM (RDP in and run it there — `MODE="local"`; SSH is optional), stages the submitted models,
runs the **C – Modeling Review** items of the rubric, and writes:

* a copy of the rubric workbook with *TWDB PM Draft Review* (Included Y/N + Comment) filled in, and
* an HTML report with the evidence and compute-log excerpts.

**Inputs:** the rubric `.xlsx`; the model submission `.zip` (download it from SharePoint first — the notebook cannot sign in to SharePoint) or a folder already on the VM.
**Outputs:** `outputs/rubric-review/<date>-<host>/` (git-ignored).
**Software:** the VM's installed HEC-RAS / HEC-HMS (versions are recorded in the report). Run `pip install -e ".[rubric]"` first.

**Result meanings:** `Y`/`N` are automated findings. A blank *Included* cell with a comment means a human must decide
(for example, comparing events against the Scope of Work). Nothing is written back to SharePoint or to the VM's original files; models are extracted into a scratch folder on the VM.

**Prerequisite (local mode):** on the VM, Python 3.10+, this repo, and `pip install openpyxl jupyter`. SSH mode additionally needs `paramiko`, port 22 reachable, and the Windows OpenSSH server (port 22 timed out from a laptop, so use local mode).

In [ ]:
import sys
from datetime import date
from pathlib import Path

REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO))  # use the checkout when the package is not pip-installed

from rubric_review import ReviewOptions, fill_workbook, load_modeling_checklist, run_review, write_html_report
from rubric_review.local import LocalSession

## What each rubric item tests

| Rubric item | Test | You must provide |
|---|---|---|
| **Model runs with no errors** | Plan's geometry + flow files exist. With `RUN_MODELS=True`: compute the plan/run, then check timeout, exit code, fresh results file, and error lines in the compute messages. | Nothing (HEC installs are detected). |
| **All events in the Scope of Work are represented** | HMS: every expected event has a simulation run. RAS: every expected event is named in the plan title, flow title or profile names (`10-yr`, `100 year`, and `1%` forms all match). The report shows a model × event matrix. | `EXPECTED_EVENTS` — the event list from the Scope of Work. |
| **Project Area in the Grant Agreement is represented** | RAS: the project-area shapefile is referenced in the `.rasmap` **and** ≥ `MIN_COVERAGE` of its area lies inside the model footprint (read from the geometry HDF). A map figure is saved per model. HMS: shapefile reference only (no geographic check yet). | `PROJECT_AREA_SHP` — the Grant Agreement shapefile, on the VM. Footprint needs `conda install geopandas h5py matplotlib`. |

Anything that cannot be decided automatically is left blank for the reviewer with the evidence in the comment.

## 1. Parameters

In [ ]:
MODE = "local"                         # "local": notebook runs ON the Windows VM (use after you RDP in).
                                       # "ssh": notebook runs elsewhere and logs in over SSH (needs port 22 + OpenSSH on the VM).
HOST = "dynamo02.tacc.utexas.edu"      # dynamo02 ... dynamo08 (label for reports; also the SSH target in ssh mode)
USERNAME = "dynamo"                    # SSH mode only

RUBRIC = REPO / "docs/rubrics/FPPG_Project_Deliverables Checklist_FINAL.xlsx"

# Provide ONE of these:
LOCAL_MODELS_ZIP = None                # e.g. Path.home() / "Downloads/Test1_10_26_2026.zip"  (extracted into a scratch folder on the VM)
VM_MODELS_ROOT = None                  # e.g. r"C:\Users\dynamo\Downloads\Test1_10_26_2026\Test1_10_26_2026" (already extracted)

RUN_MODELS = False                     # False: locate projects + check dependencies. True: also compute every plan/run (slow).
RAS_VERSION = "6.6"                    # preferred HEC-RAS version; falls back to the highest installed
HMS_VERSION = None                     # None = highest installed HEC-HMS
EXPECTED_EVENTS = ["10-yr", "25-yr", "50-yr", "100-yr"]   # EDIT to match the Scope of Work; [] = evidence only
PROJECT_AREA_SHP = None                # e.g. Path(r"C:\Users\dynamo\Downloads\GrantArea\ProjectArea.shp"); None = map-layer check only
MIN_COVERAGE = 0.95                    # share of the project area that must fall inside the model footprint
TIMEOUT_MIN = 60                       # per plan/run

OUT_DIR = REPO / "outputs/rubric-review" / f"{date.today():%Y-%m-%d}-{HOST.split('.')[0]}"
groups = load_modeling_checklist(RUBRIC)
print(f"{len(groups)} model runs / {sum(len(g.items) for g in groups)} checklist items loaded")
for g in groups:
    print(f"  {g.tool.upper():3} {g.model_name}  |  {g.run_name}")

## 2. Connect

`local` mode needs no login: your RDP session (and its MFA) is the authentication. `ssh` mode prompts for password then TACC token, input hidden, held in memory only.

In [ ]:
if MODE == "local":
    session = LocalSession()
    print("Running locally on", session.run_script("inventory_software.ps1")["computer"])
elif MODE == "ssh":
    from rubric_review.remote import VMSession
    session = VMSession(HOST, USERNAME).connect()
    print("Connected to", HOST)
else:
    raise ValueError("MODE must be 'local' or 'ssh'")

## 3. Stage the models on the VM

In [ ]:
if (LOCAL_MODELS_ZIP is None) == (VM_MODELS_ROOT is None):
    raise ValueError("Set exactly one of LOCAL_MODELS_ZIP or VM_MODELS_ROOT")

if LOCAL_MODELS_ZIP is not None:
    if MODE == "local":
        zip_on_vm, dest = str(LOCAL_MODELS_ZIP), str(Path.home() / "rubric-review" / f"models-{date.today():%Y%m%d}")
    else:
        zip_on_vm, dest = session.upload(LOCAL_MODELS_ZIP, "models.zip"), f"rubric-review\\models-{date.today():%Y%m%d}"
    staged = session.run_script("stage_models.ps1", timeout=1800, Zip=zip_on_vm, Dest=dest)
    models_root = staged["root"]
    print(f"Extracted {staged['fileCount']} files to {models_root}")
else:
    models_root = VM_MODELS_ROOT
    print("Using models already on the VM:", models_root)

## 4. Run the rubric

In [ ]:
geo = None
if PROJECT_AREA_SHP is not None:
    if MODE != "local":
        print("Footprint check needs local mode (reads model HDF files directly); falling back to the map-layer check.")
    else:
        from rubric_review.geo import LocalGeo
        geo = LocalGeo(PROJECT_AREA_SHP, OUT_DIR / "figures", MIN_COVERAGE)

options = ReviewOptions(
    models_root=models_root, run_models=RUN_MODELS, ras_version=RAS_VERSION, hms_version=HMS_VERSION,
    project_area_shapefile=Path(PROJECT_AREA_SHP).name if PROJECT_AREA_SHP else None,
    timeout_min=TIMEOUT_MIN, expected_events=EXPECTED_EVENTS,
)
outcome = run_review(session.run_script, groups, options, geo=geo)
print(outcome.environment)

## 5. Write the report

In [ ]:
results = {r.row: (r.status, r.comment) for r in outcome.results}
xlsx = fill_workbook(RUBRIC, OUT_DIR / RUBRIC.name.replace(".xlsx", "_REVIEWED.xlsx"), results)
html = write_html_report(outcome, OUT_DIR / "report.html", "FPPG Project Deliverables — Modeling Review", HOST)
print("Workbook:", xlsx)
print("Report:  ", html)

from IPython.display import Image, display
for r in outcome.results:
    if r.figure:
        display(Image(r.figure, width=420))
for r in outcome.results:
    print(f"{r.item_no:4} {r.status or '??':3} {r.kind:7} {r.model[:40]:40} {r.run:8} {r.comment[:90]}")

In [ ]:
session.close()  # drops the SSH connection and the in-memory password